# Features — Ingestion & Diagnostics

This notebook exercises the waterfall-driven ingestion path.  Key invariants
demonstrated here:

- A **zero-config** collection ingests features using code defaults (permissive `CollectionSchema`, PG routing, `on_conflict=update`).
- Policy-driven rejections return an **`IngestionReport`** — HTTP 207 for partial, 200 for fully accepted.  The report points back to the effective `CollectionWritePolicy`.
- Switching the platform default cascades to every collection without explicit overrides — the waterfall is live.

## Scenarios

1. Happy path — explicit schema + single feature.
2. **Zero-schema** collection — default resolves; feature ingests.
3. **Policy-driven rejection** — duplicate external_id returns 207 + `IngestionReport`.
4. **Bulk partial** — 10-feature batch, some rejected.
5. **Platform default switch** — change write-policy platform default, behaviour shifts everywhere.

In [ ]:
import os
import httpx

BASE = os.environ.get("DYNASTORE_BASE_URL") or "http://localhost:8080"
CATALOG_ID = "demo-ingest"

client = httpx.AsyncClient(base_url=BASE, timeout=30)

def _check(r, label=""):
    status = r.status_code
    ok = status < 400
    suffix = "" if ok else f" — {r.text[:300]}"
    print(f"{label + ': ' if label else ''}{status}{suffix}")
    return r

r = await client.post("/stac/catalogs", json={
    "id": CATALOG_ID,
    "title": "Ingestion Demo",
    "description": "Catalog for feature-ingestion and write-policy diagnostics.",
})
_check(r, "Catalog")

---
## 1. Happy path

Explicit schema, one feature, one successful response.

In [ ]:
COLL_HAPPY = "happy"

r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLL_HAPPY,
    "title": "Happy path collection",
    "description": "Collection with explicit schema for happy-path ingestion.",
    "license": "proprietary",
    "extent": {"spatial": {"bbox": [[-180, -90, 180, 90]]}, "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]}},
    "collection:schema": {
        "fields": [
            {"name": "name", "data_type": "string", "required": True},
            {"name": "temperature", "data_type": "double"},
        ],
    },
})
_check(r, "Collection with explicit schema")

feature = {
    "type": "Feature",
    "id": "f-happy-1",
    "geometry": {"type": "Point", "coordinates": [12.49, 41.89]},
    "properties": {"name": "rome-1", "temperature": 19.2},
}
r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_HAPPY}/items", json=feature)
_check(r, "Ingest")

r = await client.get(f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_HAPPY}/items")
print("Stored count:", len(r.json().get("features", [])))

---
## 2. Zero-schema collection

Create a collection with **no** explicit `CollectionSchema`.  The waterfall
resolves the code default (permissive: no fields, no constraints) and the
ingest still works — no 404 on missing `LayerConfig`, no silent drop.

In [ ]:
COLL_ZERO = "zero-schema"

r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLL_ZERO,
    "title": "Zero-config collection",
    "description": "Collection with no explicit schema — everything resolves from defaults.",
    "license": "proprietary",
    "extent": {"spatial": {"bbox": [[-180, -90, 180, 90]]}, "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]}},
})
_check(r, "Collection (no schema / no routing)")

feature = {
    "type": "Feature",
    "id": "f-zero-1",
    "geometry": {"type": "Point", "coordinates": [2.35, 48.86]},
    "properties": {"arbitrary": "value", "n": 42},
}
r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_ZERO}/items", json=feature)
_check(r, "Ingest (waterfall-defaulted)")

r = await client.get(f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_ZERO}/items")
print("Stored count:", len(r.json().get("features", [])))

---
## 3. Policy-driven rejection — IngestionReport

Set `CollectionWritePolicy.on_conflict = refuse` at collection scope.  Submit
one feature twice.  The second call returns HTTP 207 with an `IngestionReport`
whose `rejections[]` entry carries:

- the matched `geoid`,
- the `external_id`,
- the `matcher` that fired,
- a pointer to the effective policy (`/configs/.../collection:write_policy/effective`).

In [ ]:
COLL_REFUSE = "refuse-report"

r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLL_REFUSE,
    "title": "Refuse with report",
    "description": "Collection with REFUSE policy to exercise IngestionReport diagnostics.",
    "license": "proprietary",
    "extent": {"spatial": {"bbox": [[-180, -90, 180, 90]]}, "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]}},
    "collection:write_policy": {"on_conflict": "refuse", "external_id_field": "properties.ext"},
})
_check(r, "Collection (REFUSE policy)")

feature = {
    "type": "Feature",
    "geometry": {"type": "Point", "coordinates": [12.5, 41.9]},
    "properties": {"ext": "EXT-1", "v": 1},
}
r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_REFUSE}/items", json=feature)
_check(r, "First ingest (accepted)")

# Re-ingest the same external_id — must be refused and reported
feature["properties"]["v"] = 2
r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_REFUSE}/items", json=feature)
_check(r, "Duplicate ingest — expected 207 with IngestionReport")
if r.status_code in (200, 207):
    body = r.json()
    print("  accepted_ids:", body.get("accepted_ids"))
    print("  rejections  :", body.get("rejections"))

---
## 4. Bulk partial

Submit 10 features with 2 duplicate `external_id`s → HTTP 207 with 8 accepted
and 2 rejected.  The report carries both sets.

In [ ]:
COLL_BULK = "bulk-partial"

r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLL_BULK,
    "title": "Bulk partial",
    "description": "Bulk ingestion target with REFUSE policy for partial-success demo.",
    "license": "proprietary",
    "extent": {"spatial": {"bbox": [[-180, -90, 180, 90]]}, "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]}},
    "collection:write_policy": {"on_conflict": "refuse", "external_id_field": "properties.ext"},
})
_check(r, "Collection (bulk)")

# Seed two features whose external_ids will collide on the second batch
seed = {"type": "FeatureCollection", "features": [
    {"type": "Feature", "geometry": {"type": "Point", "coordinates": [0.0, 0.0]},
     "properties": {"ext": "SEED-1"}},
    {"type": "Feature", "geometry": {"type": "Point", "coordinates": [1.0, 1.0]},
     "properties": {"ext": "SEED-2"}},
]}
r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_BULK}/items", json=seed)
_check(r, "Seed batch")

# Second batch — 10 features, 2 duplicates (SEED-1, SEED-2), 8 new
batch = {"type": "FeatureCollection", "features": [
    {"type": "Feature", "geometry": {"type": "Point", "coordinates": [float(i), float(i)]},
     "properties": {"ext": f"EXT-{i}"}}
    for i in range(8)
] + [
    {"type": "Feature", "geometry": {"type": "Point", "coordinates": [0.0, 0.0]},
     "properties": {"ext": "SEED-1"}},
    {"type": "Feature", "geometry": {"type": "Point", "coordinates": [1.0, 1.0]},
     "properties": {"ext": "SEED-2"}},
]}
r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_BULK}/items", json=batch)
_check(r, "Bulk mixed batch — expected 207")
if r.status_code in (200, 207):
    body = r.json()
    print("  total      :", body.get("total"))
    print("  accepted   :", len(body.get("accepted_ids") or []))
    print("  rejections :", len(body.get("rejections") or []))
    print("  is_partial :", body.get("is_partial"))

---
## 5. Platform default switch

Change the **platform-scope** `CollectionWritePolicy` from `update` → `refuse`.
A collection that has no explicit policy now inherits the new default and
starts refusing duplicates — without touching its own config.  Proof that the
waterfall is live, not just read once at creation.

In [ ]:
COLL_PLATFORM = "platform-switch"

r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLL_PLATFORM,
    "title": "Platform switch demo",
    "description": "Collection with no explicit write policy — inherits platform default.",
    "license": "proprietary",
    "extent": {"spatial": {"bbox": [[-180, -90, 180, 90]]}, "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]}},
})
_check(r, "Collection (no explicit policy)")

feat = {
    "type": "Feature",
    "geometry": {"type": "Point", "coordinates": [0.0, 0.0]},
    "properties": {"ext": "DUP"},
}

# Baseline — code/platform default = update; duplicate overwrites in place
r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_PLATFORM}/items", json=feat)
_check(r, "First ingest (update default)")
r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_PLATFORM}/items", json=feat)
_check(r, "Duplicate ingest — expected 200 (overwrite)")

# Flip platform default to REFUSE
r = await client.put(
    "/configs/platform/configs/collection:write_policy",
    json={"on_conflict": "refuse", "external_id_field": "properties.ext"},
)
_check(r, "Platform default -> refuse")

# Same collection, same external_id — now refused
r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_PLATFORM}/items", json=feat)
_check(r, "Duplicate ingest — expected 207 after platform switch")
if r.status_code in (200, 207):
    body = r.json()
    print("  rejections :", body.get("rejections"))

In [ ]:
# Revert platform default so later notebooks start clean
r = await client.delete("/configs/platform/configs/collection:write_policy")
_check(r, "Revert platform write_policy")

r = await client.delete(f"/stac/catalogs/{CATALOG_ID}?force=true")
_check(r, "Cleanup catalog")
await client.aclose()